# Topic 3 - Observability-driven debugging and cost quality

**The core shift.** You cannot set a breakpoint inside a non-deterministic agent.
The reasoning happens in the model. So you debug from **traces** captured while
the agent ran, not by stepping through code afterwards. And the same run may not
reproduce, so the trace is often the only evidence you get.

**The three-tier model you read traces in:**

| Tier | What it is | Holds |
|------|------------|-------|
| Session | One customer conversation | many traces |
| Trace | One request inside the session | many spans |
| Span | One tool call or one model call | input, output, latency, tokens |

A guardrail block shows up as a span too, carrying its action and category.

**Two outputs from this notebook.** First, the skill of localising a failing run
to a single span. Second, `cost_latency.json`, which the quality gate (Topic 4)
reads. Cost here is not a finance number; a spike usually means retries or a loop,
which is a quality bug.

Read top to bottom.

## Setup

**Enable observability (one time per account and Region).** Span and trace data
will not appear until you turn on CloudWatch Transaction Search:

> CloudWatch Console -> Application Signals -> Transaction Search -> enable.

This is the single most common reason a fresh account shows empty dashboards.

**Instrument the agent.**
- On **AgentCore Runtime**, instrumentation is automatic; you do nothing.
- **Outside Runtime** (local, Lambda, ECS), add `aws-opentelemetry-distro` to
  `requirements.txt` and launch with:
  `opentelemetry-instrument python -m travelmind_agent`

Strands emits OpenTelemetry spans natively, so once instrumented, every model
call and tool call becomes a span. Spans land in the CloudWatch Logs group
`aws/spans` (see `config.SPANS_LOG_GROUP`).

**Credentials.** VS Code: `aws configure`. Colab: set `AWS_ACCESS_KEY_ID`,
`AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` from Secrets. The IAM principal
needs CloudWatch Logs read (`logs:StartQuery`, `logs:GetQueryResults`) on the
`aws/spans` group, plus Bedrock invoke to run the agent. Least privilege: grant
only those.

In [ ]:
# --- imports and clients ---
import json
import time

import boto3

import config

# CloudWatch Logs client. Logs Insights runs through start_query / get_query_results.
logs = boto3.client("logs", region_name=config.REGION)

print("region        :", config.REGION)
print("spans log grp :", config.SPANS_LOG_GROUP)

## Step 1 - generate a trace

Run the agent once so there is something to look at. We also read the token usage
straight off the result. In production the same token counts are on the span;
`result.metrics` is the convenient local source while you are developing.

In [ ]:
from travelmind_agent import get_agent

def tokens_from_result(result):
    """Pull input and output token counts off a Strands result.

    Strands exposes usage on result.metrics. Attribute names have shifted across
    versions, so we read defensively and accept either camelCase or snake_case.
    If you get zeros, print(result.metrics) once to see the exact shape in your
    installed version, then adjust. Honest beats guessing."""
    m = getattr(result, "metrics", None)
    usage = getattr(m, "accumulated_usage", None) or {}
    in_tok = usage.get("inputTokens") or usage.get("input_tokens") or 0
    out_tok = usage.get("outputTokens") or usage.get("output_tokens") or 0
    return in_tok, out_tok

# Example (uncomment when AWS is configured):
# result = get_agent()("My flight on PNR JX48Q2 was cancelled. What are my options?")
# print(str(result))
# print("tokens (in, out):", tokens_from_result(result))

## Step 2 - query the spans

CloudWatch Logs Insights is how you search `aws/spans`. The helper below starts a
query, polls until it completes, and returns each row as a plain dict so it is
easy to work with.

The first query we care about jumps straight to **failed tool spans**: tool calls
whose status is an error. That is where a swallowed tool failure shows itself.

A caveat worth stating: the exact field paths inside `aws/spans` follow the
OpenTelemetry GenAI semantic conventions and can differ slightly by setup. Run a
broad `fields @message | limit 5` first to see your real field names, then refine.

In [ ]:
def run_insights_query(query: str, minutes: int = 60,
                       log_group: str = None) -> list:
    """Run a CloudWatch Logs Insights query and return rows as dicts. Needs AWS."""
    log_group = log_group or config.SPANS_LOG_GROUP
    end = int(time.time())
    start = end - minutes * 60                  # look back this many minutes

    query_id = logs.start_query(
        logGroupName=log_group,
        startTime=start,
        endTime=end,
        queryString=query,
    )["queryId"]

    # Logs Insights is asynchronous: start it, then poll for completion.
    while True:
        resp = logs.get_query_results(queryId=query_id)
        if resp["status"] in ("Complete", "Failed", "Cancelled"):
            break
        time.sleep(1)

    # Each row is a list of {"field": ..., "value": ...}; flatten to a dict.
    return [{col["field"]: col["value"] for col in row} for row in resp["results"]]


# Query: the most recent failed tool spans.
FAILED_TOOL_SPANS = """
fields @timestamp, name, durationNano, status.code
| filter name like /tool/
| filter status.code = "ERROR"
| sort @timestamp desc
| limit 20
"""

# Example (uncomment when AWS is configured):
# for row in run_insights_query(FAILED_TOOL_SPANS):
#     print(row)

## Step 3 - worked example: localise a failing run

A customer reports TravelMind confirmed a rebooking that never happened. You have
the session id. The walkthrough:

1. Pull every span for that session, oldest first.
2. Find the span with an error status. That names the failing tool.
3. Read its input and the tokens spent, so you know what was attempted and what
   it cost before it failed.

The query is parameterised by session id. `find_failing_span` then picks the
first error span out of the rows.

In [ ]:
def trace_for_session(session_id: str) -> list:
    """All spans for one session, oldest first. Needs AWS."""
    query = f"""
    fields @timestamp, name, status.code, durationNano
    | filter sessionId = "{session_id}"
    | sort @timestamp asc
    | limit 100
    """
    return run_insights_query(query)


def find_failing_span(rows: list) -> dict:
    """Return the first span whose status is an error, or {} if all passed.
    Works on whatever run_insights_query returned, so it is testable on sample
    rows without AWS (see the assertion below)."""
    for row in rows:
        if str(row.get("status.code", "")).upper() in ("ERROR", "STATUS_CODE_ERROR"):
            return row
    return {}


# Pure-logic self-check on sample rows (no AWS): the failing tool is found.
_sample = [
    {"name": "model.invoke", "status.code": "OK"},
    {"name": "tool.get_rebooking_options", "status.code": "ERROR"},
    {"name": "model.invoke", "status.code": "OK"},
]
assert find_failing_span(_sample)["name"] == "tool.get_rebooking_options"
print("find_failing_span works on sample rows")

# Example (uncomment when AWS is configured):
# rows = trace_for_session("sess-8841")
# print("failing span:", find_failing_span(rows))

## Step 4 - cost as a quality signal

Cost is computed from tokens and the per-model price (both in `config.py`). The
reason it belongs in a QA notebook: a sudden jump in cost per resolution almost
always means the agent took more turns than usual, and more turns usually means
retries or a loop. So you alarm on cost the same way you alarm on errors.

We write `cost_latency.json` here for the gate. The p95 latency would come from
your metrics; we pass a representative value.

In [ ]:
def run_cost(in_tok: int, out_tok: int, price_key: str = None) -> float:
    """Cost of one run in USD = input and output tokens at the per-model price."""
    price_key = price_key or config.AGENT_PRICE_KEY
    price_in, price_out = config.PRICES[price_key]          # USD per 1M tokens
    return in_tok / 1e6 * price_in + out_tok / 1e6 * price_out


# Pure-logic self-check (no AWS): a known run costs the expected amount.
# 4200 in * $1/1M  +  380 out * $5/1M  =  0.0042 + 0.0019 = 0.0061
assert abs(run_cost(4200, 380) - 0.0061) < 1e-9
print("run_cost is correct: 4200 in / 380 out ->", round(run_cost(4200, 380), 5), "USD")


def write_cost_latency(cost_usd: float, p95_ms: int, path: str = "cost_latency.json"):
    """Persist the cost and latency numbers the gate will read."""
    payload = {"cost_usd": round(cost_usd, 5), "p95_ms": p95_ms}
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"wrote {path}: {payload}")

# Typical resolution, written for the gate:
write_cost_latency(run_cost(4200, 380), p95_ms=3120)

## What changes in production

- **Run all four signals together**: metrics, logs, traces, and quality. Three
  are technical; quality is the product signal. An agent can be up, fast, and
  wrong.
- **Alarm on cost per resolution and p95 latency**, not only on error rate. A
  cost spike is the cheapest early warning of a looping agent.
- **Add a few custom spans at your own risk points**, but do not dump everything;
  a noisy trace hides the failing span you are looking for.
- **Read tokens from the span in production**, not from a local result object;
  the span is the durable, queryable source.
- **Keep a triage runbook**: tool failure vs model failure vs orchestration. The
  failing span usually tells you which, in seconds.